# M3 Profiling — NYC 311 Service Requests and Open-Meteo Weather

**Lifecycle stage:** CLEAN  
**Database:** `data/nyc311_weather.db`  
**Purpose:** Profile the M2 tables before making any cleaning decisions.

This notebook is read-only. It does not update, delete, or create tables in the M2 database. It profiles missingness, value ranges, categories, time coverage, sequences, cross-field consistency, and identity keys.

Run every cell from top to bottom from a fresh M2 database build. Keep the executed notebook outputs when committing `notebooks/m3_profile.ipynb`.

**Important:** An IQR candidate or chronological inconsistency is an investigation signal, not automatic proof that a value is wrong. Unusual but possible weather events must be retained.


## 1. Connect to the M2 database

The connection uses SQLite read-only mode. If the database does not exist, build it first with `python3 src/load.py`.


In [1]:
from pathlib import Path
import sqlite3
from collections import Counter
from difflib import SequenceMatcher
from statistics import median

CANDIDATES = [Path.cwd(), Path.cwd().parent]
ROOT = next(
    (path.resolve() for path in CANDIDATES
     if (path / "data" / "nyc311_weather.db").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Could not find data/nyc311_weather.db. "
        "Start Jupyter from the project root or its notebooks folder."
    )

DB_PATH = ROOT / "data" / "nyc311_weather.db"
con = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
con.row_factory = sqlite3.Row

print(f"Project root: {ROOT}")
print(f"Database: {DB_PATH}")
print("SQLite read-only connection: OPEN")

def rows(sql, parameters=()):
    return con.execute(sql, parameters).fetchall()

def display_rows(title, result_rows, limit=100):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)
    for row in result_rows[:limit]:
        print(tuple(row))
    if len(result_rows) > limit:
        print(f"... {len(result_rows) - limit:,} additional rows omitted")

def quantile(sorted_values, probability):
    if not sorted_values:
        return None
    position = (len(sorted_values) - 1) * probability
    lower = int(position)
    upper = min(lower + 1, len(sorted_values) - 1)
    fraction = position - lower
    return (
        sorted_values[lower] * (1 - fraction)
        + sorted_values[upper] * fraction
    )


Project root: /home/seymanur/projects/nyc-311-weather-analysis
Database: /home/seymanur/projects/nyc-311-weather-analysis/data/nyc311_weather.db
SQLite read-only connection: OPEN


## 2. Table inventory and schema properties

The row count and `STRICT` status checks establish the starting point for the cleaned layer. The M2 row counts must remain unchanged after `clean.py` is run.


In [2]:
TABLES = [
    "agencies", "boroughs", "complaint_types", "request_statuses",
    "submission_channels", "weather_locations", "service_requests",
    "weather_observations", "quarantine_records",
]

inventory = []
table_list = {
    row["name"]: row
    for row in rows("PRAGMA table_list")
    if row["schema"] == "main"
}
for table in TABLES:
    count = rows(f'SELECT COUNT(*) AS n FROM "{table}"')[0]["n"]
    strict = table_list[table]["strict"]
    inventory.append((table, count, bool(strict)))

display_rows("M2 TABLE INVENTORY: table, row count, STRICT", inventory)



M2 TABLE INVENTORY: table, row count, STRICT
('agencies', 16, True)
('boroughs', 6, True)
('complaint_types', 194, True)
('request_statuses', 7, True)
('submission_channels', 5, True)
('weather_locations', 5, True)
('service_requests', 3953178, True)
('weather_observations', 43800, True)
('quarantine_records', 0, True)


## 3. Missing values

Missingness is profiled overall and by observable entities/statuses. A missing value is not automatically invalid: for example, an unresolved request may not yet have a closure or resolution timestamp.


In [3]:
request_columns = [
    "latitude", "longitude",
    "x_coordinate_state_plane", "y_coordinate_state_plane",
    "created_at_utc", "closed_date_source",
    "due_date_source", "resolution_action_updated_date_source",
]

missing_select = ", ".join(
    f'SUM("{column}" IS NULL) AS "{column}_missing"'
    for column in request_columns
)
overall_missing = rows(
    f"SELECT COUNT(*) AS total, {missing_select} FROM service_requests"
)
display_rows("SERVICE REQUEST MISSING VALUES: overall", overall_missing)

display_rows("MISSING VALUES BY BOROUGH", rows("""
    SELECT b.borough_name,
           COUNT(*) AS total,
           SUM(r.latitude IS NULL) AS missing_latitude,
           SUM(r.longitude IS NULL) AS missing_longitude,
           SUM(r.due_date_source IS NULL) AS missing_due_date,
           SUM(r.closed_date_source IS NULL) AS missing_closed_date,
           SUM(r.resolution_action_updated_date_source IS NULL)
               AS missing_resolution_update
    FROM service_requests r
    JOIN boroughs b ON b.borough_id = r.borough_id
    GROUP BY b.borough_name
    ORDER BY b.borough_name
"""))

display_rows("MISSING VALUES BY REQUEST STATUS", rows("""
    SELECT s.status_name,
           COUNT(*) AS total,
           SUM(r.closed_date_source IS NULL) AS missing_closed,
           SUM(r.due_date_source IS NULL) AS missing_due,
           SUM(r.resolution_action_updated_date_source IS NULL)
               AS missing_resolution_update
    FROM service_requests r
    JOIN request_statuses s ON s.status_id = r.status_id
    GROUP BY s.status_name
    ORDER BY total DESC
"""))

display_rows("MONTHLY MISSINGNESS BY BOROUGH", rows("""
    SELECT b.borough_name,
           substr(r.created_local_date, 1, 7) AS month,
           COUNT(*) AS total,
           SUM(r.latitude IS NULL) AS missing_latitude,
           SUM(r.due_date_source IS NULL) AS missing_due_date,
           SUM(r.resolution_action_updated_date_source IS NULL)
               AS missing_resolution_update
    FROM service_requests r
    JOIN boroughs b ON b.borough_id = r.borough_id
    GROUP BY b.borough_name, month
    ORDER BY b.borough_name, month
"""), limit=100)

display_rows("DUE DATE MISSINGNESS BY AGENCY AND STATUS", rows("""
    SELECT a.agency_code, s.status_name,
           COUNT(*) AS total,
           SUM(r.due_date_source IS NULL) AS missing,
           ROUND(100.0 * SUM(r.due_date_source IS NULL) / COUNT(*), 2)
               AS missing_pct
    FROM service_requests r
    JOIN agencies a ON a.agency_id = r.agency_id
    JOIN request_statuses s ON s.status_id = r.status_id
    GROUP BY a.agency_code, s.status_name
    ORDER BY total DESC
"""), limit=100)

display_rows("NON-NULL DUE DATE RECORDS BY AGENCY, CATEGORY AND STATUS", rows("""
    SELECT a.agency_code, c.complaint_type, s.status_name,
           COUNT(*) AS records_with_due_date
    FROM service_requests r
    JOIN agencies a ON a.agency_id = r.agency_id
    JOIN complaint_types c ON c.complaint_type_id = r.complaint_type_id
    JOIN request_statuses s ON s.status_id = r.status_id
    WHERE r.due_date_source IS NOT NULL
    GROUP BY a.agency_code, c.complaint_type, s.status_name
    ORDER BY records_with_due_date DESC
"""))

display_rows("TIMESTAMP QUALITY STATUS COUNTS", [
    (column, row["status"], row["n"])
    for column in [
        "created_timestamp_status", "closed_timestamp_status",
        "due_timestamp_status", "resolution_updated_timestamp_status",
    ]
    for row in rows(
        f'SELECT "{column}" AS status, COUNT(*) AS n '
        f'FROM service_requests GROUP BY "{column}" ORDER BY n DESC'
    )
])



SERVICE REQUEST MISSING VALUES: overall
(3953178, 82051, 82051, 81974, 81638, 797, 179314, 3938171, 69788)

MISSING VALUES BY BOROUGH
('BRONX', 842817, 10274, 10274, 841188, 25945, 5712)
('BROOKLYN', 1212961, 18040, 18040, 1205711, 51477, 25978)
('MANHATTAN', 768500, 11523, 11523, 764690, 43430, 18633)
('QUEENS', 963957, 33325, 33325, 961969, 49577, 14477)
('STATEN ISLAND', 160985, 5205, 5205, 160655, 7933, 4525)
('UNSPECIFIED', 3958, 3684, 3684, 3958, 952, 463)

MISSING VALUES BY REQUEST STATUS
('CLOSED', 3752558, 15, 3738981, 0)
('IN PROGRESS', 115283, 113409, 115283, 64499)
('OPEN', 65567, 56363, 64137, 5266)
('ASSIGNED', 12853, 3159, 12853, 0)
('PENDING', 5457, 5064, 5457, 11)
('STARTED', 1282, 1275, 1282, 0)
('UNSPECIFIED', 178, 29, 178, 12)

MONTHLY MISSINGNESS BY BOROUGH
('BRONX', '2025-10', 53411, 315, 53311, 106)
('BRONX', '2025-11', 74674, 339, 74477, 120)
('BRONX', '2025-12', 96322, 377, 96202, 81)
('BRONX', '2026-01', 77457, 504, 77326, 142)
('BRONX', '2026-02', 70060, 702

## 4. Numeric ranges and robust outlier candidates

The request coordinates are checked against geographic bounds and for zero/negative state-plane sentinel candidates. Weather values are profiled by borough. IQR fences are calculated per borough; precipitation is also examined among positive values only because a zero-heavy distribution makes the ordinary all-hours IQR fence uninformative.

**Do not delete IQR candidates.** Inspect the timestamps and retain plausible weather events.


In [4]:
display_rows("REQUEST COORDINATE RANGES BY BOROUGH", rows("""
    SELECT b.borough_name,
           COUNT(r.latitude) AS records_with_latitude,
           MIN(r.latitude) AS min_latitude,
           MAX(r.latitude) AS max_latitude,
           MIN(r.longitude) AS min_longitude,
           MAX(r.longitude) AS max_longitude,
           SUM(r.latitude < -90 OR r.latitude > 90) AS invalid_latitude,
           SUM(r.longitude < -180 OR r.longitude > 180) AS invalid_longitude
    FROM service_requests r
    JOIN boroughs b ON b.borough_id = r.borough_id
    GROUP BY b.borough_name
    ORDER BY b.borough_name
"""))

display_rows("STATE-PLANE COORDINATE CHECKS", rows("""
    SELECT
        COUNT(*) AS total,
        SUM(x_coordinate_state_plane < 0) AS negative_x,
        SUM(y_coordinate_state_plane < 0) AS negative_y,
        SUM(x_coordinate_state_plane = 0) AS zero_x,
        SUM(y_coordinate_state_plane = 0) AS zero_y,
        MIN(x_coordinate_state_plane) AS min_x,
        MAX(x_coordinate_state_plane) AS max_x,
        MIN(y_coordinate_state_plane) AS min_y,
        MAX(y_coordinate_state_plane) AS max_y
    FROM service_requests
"""))

weather_variables = [
    "temperature_2m_c", "relative_humidity_2m_pct",
    "precipitation_mm", "wind_speed_10m_kmh",
    "wind_direction_10m_degrees", "cloud_cover_pct",
]

for variable in weather_variables:
    print(f"\nWEATHER IQR PROFILE: {variable}")
    for borough_row in rows("""
        SELECT borough_name FROM boroughs
        WHERE borough_name != 'UNSPECIFIED'
        ORDER BY borough_name
    """):
        borough = borough_row["borough_name"]
        values = [
            item["value"]
            for item in rows(
                f"""
                SELECT w."{variable}" AS value
                FROM weather_observations w
                JOIN weather_locations l
                  ON l.weather_location_id = w.weather_location_id
                JOIN boroughs b ON b.borough_id = l.borough_id
                WHERE b.borough_name = ?
                  AND w."{variable}" IS NOT NULL
                ORDER BY w."{variable}"
                """,
                (borough,),
            )
        ]
        if not values:
            print(borough, "NO VALUES")
            continue
        q1 = quantile(values, 0.25)
        q3 = quantile(values, 0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        candidates = sum(value < lower or value > upper for value in values)
        mode_value, mode_count = Counter(values).most_common(1)[0]
        print({
            "borough": borough,
            "count": len(values),
            "min": values[0],
            "median": median(values),
            "max": values[-1],
            "mode": mode_value,
            "mode_count": mode_count,
            "q1": round(q1, 3),
            "q3": round(q3, 3),
            "lower_fence": round(lower, 3),
            "upper_fence": round(upper, 3),
            "candidates": candidates,
        })

for borough_row in rows("""
    SELECT borough_name FROM boroughs
    WHERE borough_name != 'UNSPECIFIED'
    ORDER BY borough_name
"""):
    borough = borough_row["borough_name"]
    values = [
        item["value"]
        for item in rows("""
            SELECT w.precipitation_mm AS value
            FROM weather_observations w
            JOIN weather_locations l
              ON l.weather_location_id = w.weather_location_id
            JOIN boroughs b ON b.borough_id = l.borough_id
            WHERE b.borough_name = ?
              AND w.precipitation_mm > 0
            ORDER BY w.precipitation_mm
        """, (borough,))
    ]
    q1, q3 = quantile(values, 0.25), quantile(values, 0.75)
    upper = q3 + 1.5 * (q3 - q1)
    print({
        "positive_precipitation_borough": borough,
        "positive_hours": len(values),
        "q1": round(q1, 3),
        "q3": round(q3, 3),
        "upper_fence": round(upper, 3),
        "upper_candidates": sum(value > upper for value in values),
        "maximum_mm": max(values),
    })



REQUEST COORDINATE RANGES BY BOROUGH
('BRONX', 832543, 40.79030103908103, 40.912868795316655, -73.93604025211081, -73.78233492516738, 0, 0)
('BROOKLYN', 1194921, 40.57148260848523, 40.76060052051626, -74.04271671853803, -73.85575819288555, 0, 0)
('MANHATTAN', 756977, 40.685308843305485, 40.87788917541197, -74.02123221319597, -73.84992636527032, 0, 0)
('QUEENS', 930632, 40.552625521844114, 40.79989611634838, -73.96185005320459, -73.70036545591262, 0, 0)
('STATEN ISLAND', 155780, 40.49888624741073, 40.648357171854954, -74.25495171973925, -74.04420370181288, 0, 0)
('UNSPECIFIED', 274, 40.557372578559225, 40.87436101316596, -74.16983111438478, -73.8226204874806, 0, 0)

STATE-PLANE COORDINATE CHECKS
(3953178, 0, 0, 0, 0, 913353.0, 1067284.0, 121127.0, 271876.0)

WEATHER IQR PROFILE: temperature_2m_c
{'borough': 'BRONX', 'count': 8760, 'min': -17.6, 'median': 13.0, 'max': 38.9, 'mode': 22.4, 'mode_count': 47, 'q1': 2.9, 'q3': 21.6, 'lower_fence': -25.15, 'upper_fence': 49.65, 'candidates': 

## 5. Categories and text representation

`repr()` helps expose whitespace and special-character differences. Similarity scores are candidate-generation only; they do not authorize merging categories.


In [5]:
category_columns = [
    ("boroughs", "borough_name"),
    ("agencies", "agency_code"),
    ("agencies", "agency_name"),
    ("complaint_types", "complaint_type"),
    ("request_statuses", "status_name"),
    ("submission_channels", "channel_name"),
    ("weather_locations", "location_code"),
]

for table, column in category_columns:
    values = [
        row[0] for row in rows(
            f'SELECT "{column}" FROM "{table}" ORDER BY "{column}"'
        )
    ]
    print(f"\n{table}.{column}: {len(values)} distinct values")
    for value in values:
        print(repr(value))

categories = [
    row[0] for row in rows(
        "SELECT complaint_type FROM complaint_types ORDER BY complaint_type"
    )
]
similar_pairs = []
for index, first in enumerate(categories):
    for second in categories[index + 1:]:
        score = SequenceMatcher(None, first, second).ratio()
        if score >= 0.85:
            similar_pairs.append((round(score, 3), first, second))

display_rows(
    "SIMILAR CATEGORY NAME CANDIDATES (REVIEW ONLY)",
    sorted(similar_pairs, reverse=True),
    limit=100,
)



boroughs.borough_name: 6 distinct values
'BRONX'
'BROOKLYN'
'MANHATTAN'
'QUEENS'
'STATEN ISLAND'
'UNSPECIFIED'

agencies.agency_code: 16 distinct values
'DCWP'
'DEP'
'DHS'
'DOB'
'DOE'
'DOHMH'
'DOT'
'DPR'
'DSNY'
'EDC'
'HPD'
'NYC311-PRD'
'NYPD'
'OOS'
'OTI'
'TLC'

agencies.agency_name: 16 distinct values
'Department of Buildings'
'Department of Consumer and Worker Protection'
'Department of Education'
'Department of Environmental Protection'
'Department of Health and Mental Hygiene'
'Department of Homeless Services'
'Department of Housing Preservation and Development'
'Department of Parks and Recreation'
'Department of Sanitation'
'Department of Transportation'
'Economic Development Corporation'
'HIQA'
'New York City Police Department'
'Office of Technology and Innovation'
'Office of the Sheriff'
'Taxi and Limousine Commission'

complaint_types.complaint_type: 194 distinct values
'ABANDONED BIKE'
'ABANDONED VEHICLE'
'AHV INSPECTION UNIT'
'AIR QUALITY'
'ANIMAL FACILITY - NO PERMIT'
'ANIMA

## 6. Time coverage and sequence checks

This section checks first/last timestamps, monthly coverage, chronological consistency between fields, and long runs of identical hourly weather values.


In [6]:
display_rows("REQUEST COVERAGE BY BOROUGH", rows("""
    SELECT b.borough_name,
           COUNT(*) AS requests,
           MIN(r.created_local_date) AS first_date,
           MAX(r.created_local_date) AS last_date,
           COUNT(DISTINCT r.created_local_date) AS active_days
    FROM service_requests r
    JOIN boroughs b ON b.borough_id = r.borough_id
    GROUP BY b.borough_name
    ORDER BY b.borough_name
"""))

display_rows("MONTHLY REQUEST COUNTS BY BOROUGH", rows("""
    SELECT b.borough_name,
           substr(r.created_local_date, 1, 7) AS month,
           COUNT(*) AS requests
    FROM service_requests r
    JOIN boroughs b ON b.borough_id = r.borough_id
    GROUP BY b.borough_name, month
    ORDER BY b.borough_name, month
"""), limit=100)

display_rows("WEATHER COVERAGE BY BOROUGH", rows("""
    SELECT b.borough_name,
           COUNT(*) AS observations,
           MIN(w.observed_at_utc) AS first_utc,
           MAX(w.observed_at_utc) AS last_utc,
           COUNT(DISTINCT w.observed_at_utc) AS distinct_hours
    FROM weather_observations w
    JOIN weather_locations l
      ON l.weather_location_id = w.weather_location_id
    JOIN boroughs b ON b.borough_id = l.borough_id
    GROUP BY b.borough_name
    ORDER BY b.borough_name
"""))

display_rows("CROSS-FIELD CHRONOLOGY COUNTS", rows("""
    SELECT
        SUM(closed_at_utc < created_at_utc) AS closed_before_created,
        SUM(due_at_utc < created_at_utc) AS due_before_created,
        SUM(resolution_action_updated_at_utc < created_at_utc)
            AS resolution_before_created
    FROM service_requests
    WHERE created_at_utc IS NOT NULL
"""))

display_rows("CLOSED BEFORE CREATED BY AGENCY", rows("""
    SELECT a.agency_code, COUNT(*) AS affected
    FROM service_requests r
    JOIN agencies a ON a.agency_id = r.agency_id
    WHERE r.closed_at_utc < r.created_at_utc
    GROUP BY a.agency_code
    ORDER BY affected DESC
"""))

display_rows("RESOLUTION UPDATE BEFORE CREATED BY AGENCY", rows("""
    SELECT a.agency_code, COUNT(*) AS affected
    FROM service_requests r
    JOIN agencies a ON a.agency_id = r.agency_id
    WHERE r.resolution_action_updated_at_utc < r.created_at_utc
    GROUP BY a.agency_code
    ORDER BY affected DESC
"""))

weather_run_variables = [
    "temperature_2m_c", "precipitation_mm",
    "wind_speed_10m_kmh", "relative_humidity_2m_pct",
]

for variable in weather_run_variables:
    for borough_row in rows("""
        SELECT borough_name FROM boroughs
        WHERE borough_name != 'UNSPECIFIED'
        ORDER BY borough_name
    """):
        borough = borough_row["borough_name"]
        observations = rows(
            f"""
            SELECT w.observed_at_utc, w."{variable}" AS value
            FROM weather_observations w
            JOIN weather_locations l
              ON l.weather_location_id = w.weather_location_id
            JOIN boroughs b ON b.borough_id = l.borough_id
            WHERE b.borough_name = ?
            ORDER BY w.observed_at_utc
            """,
            (borough,),
        )

        best_length = 0
        best_value = None
        best_start = None
        best_end = None
        current_value = object()
        current_length = 0
        current_start = None
        previous_time = None

        for observation in observations:
            value = observation["value"]
            timestamp = observation["observed_at_utc"]
            if value == current_value:
                current_length += 1
            else:
                if current_length > best_length:
                    best_length = current_length
                    best_value = current_value
                    best_start = current_start
                    best_end = previous_time
                current_value = value
                current_start = timestamp
                current_length = 1
            previous_time = timestamp

        if current_length > best_length:
            best_length = current_length
            best_value = current_value
            best_start = current_start
            best_end = previous_time

        print({
            "longest_identical_run": variable,
            "borough": borough,
            "value": best_value,
            "hours": best_length,
            "from_utc": best_start,
            "to_utc": best_end,
        })



REQUEST COVERAGE BY BOROUGH
('BRONX', 842817, '2025-10-12', '2026-10-10', 364)
('BROOKLYN', 1212961, '2025-10-12', '2026-10-10', 364)
('MANHATTAN', 768500, '2025-10-12', '2026-10-10', 364)
('QUEENS', 963957, '2025-10-12', '2026-10-10', 364)
('STATEN ISLAND', 160985, '2025-10-12', '2026-10-10', 364)
('UNSPECIFIED', 3958, '2025-10-12', '2026-10-09', 363)

MONTHLY REQUEST COUNTS BY BOROUGH
('BRONX', '2025-10', 53411)
('BRONX', '2025-11', 74674)
('BRONX', '2025-12', 96322)
('BRONX', '2026-01', 77457)
('BRONX', '2026-02', 70060)
('BRONX', '2026-03', 69430)
('BRONX', '2026-04', 59817)
('BRONX', '2026-05', 61600)
('BRONX', '2026-06', 63941)
('BRONX', '2026-07', 74204)
('BRONX', '2026-08', 64306)
('BRONX', '2026-09', 59212)
('BRONX', '2026-10', 18383)
('BROOKLYN', '2025-10', 65448)
('BROOKLYN', '2025-11', 88591)
('BROOKLYN', '2025-12', 93508)
('BROOKLYN', '2026-01', 106935)
('BROOKLYN', '2026-02', 103164)
('BROOKLYN', '2026-03', 103617)
('BROOKLYN', '2026-04', 92948)
('BROOKLYN', '2026-05', 1

## 7. Identity and referential-integrity checks

The request natural key should be non-null and unique. Weather observations should be unique by `(weather_location_id, observed_at_utc)`. The foreign-key check should return no rows.


In [7]:
display_rows("REQUEST KEY CHECK", rows("""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT unique_key) AS distinct_keys,
           SUM(unique_key IS NULL) AS null_keys
    FROM service_requests
"""))

display_rows("WEATHER COMPOSITE KEY CHECK", rows("""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT weather_location_id || '|' || observed_at_utc)
               AS distinct_composite_keys
    FROM weather_observations
"""))

display_rows("DIMENSION NATURAL KEY DUPLICATE CHECKS", rows("""
    SELECT 'agency_code' AS key_name, COUNT(*) - COUNT(DISTINCT agency_code) AS duplicates
    FROM agencies
    UNION ALL
    SELECT 'borough_name', COUNT(*) - COUNT(DISTINCT borough_name) FROM boroughs
    UNION ALL
    SELECT 'complaint_type', COUNT(*) - COUNT(DISTINCT complaint_type) FROM complaint_types
    UNION ALL
    SELECT 'status_name', COUNT(*) - COUNT(DISTINCT status_name) FROM request_statuses
    UNION ALL
    SELECT 'channel_name', COUNT(*) - COUNT(DISTINCT channel_name) FROM submission_channels
"""))

display_rows("FOREIGN KEY CHECK", rows("PRAGMA foreign_key_check"))



REQUEST KEY CHECK
(3953178, 3953178, 0)

WEATHER COMPOSITE KEY CHECK
(43800, 43800)

DIMENSION NATURAL KEY DUPLICATE CHECKS
('agency_code', 0)
('borough_name', 0)
('complaint_type', 0)
('status_name', 0)
('channel_name', 0)

FOREIGN KEY CHECK


## 8. Six-family investigation register — profiling summary

Use the query outputs above to complete the issue register in `docs/cleaning_report.md`. Keep the cause classification honest:

- **Confirmed:** supported by provider documentation or a reliable log.
- **Inferred:** supported by the observed data pattern, but not explicitly stated by the provider.
- **Unknown:** the problem is visible but its cause is not established.

Do not treat an unusual but possible weather observation as invalid. Do not impute missing values in M3. Store uncertain cleaned measurements as `NULL` with an explanatory flag while preserving the raw value.


In [8]:
family_checks = [
    ("Missing values", "NULL counts by field, borough, status, month and agency"),
    ("Invalid values", "coordinate bounds, numeric domains, timestamp chronology"),
    ("Unusual values", "per-borough IQR profiles; positive-only precipitation IQR"),
    ("Failures across rows/fields", "chronology checks and longest identical-value runs"),
    ("Text and categories", "distinct values using repr(); similarity candidates for review"),
    ("Identity", "request natural key, weather composite key, dimension keys, foreign keys"),
]

for family, checks in family_checks:
    print(f"CHECKED | {family} | {checks}")

print("\nProfiling completed. Review outputs before finalising cleaning rules.")


CHECKED | Missing values | NULL counts by field, borough, status, month and agency
CHECKED | Invalid values | coordinate bounds, numeric domains, timestamp chronology
CHECKED | Unusual values | per-borough IQR profiles; positive-only precipitation IQR
CHECKED | Failures across rows/fields | chronology checks and longest identical-value runs
CHECKED | Text and categories | distinct values using repr(); similarity candidates for review
CHECKED | Identity | request natural key, weather composite key, dimension keys, foreign keys

Profiling completed. Review outputs before finalising cleaning rules.


## 9. Close the read-only connection

Run this cell after reviewing the notebook output.


In [9]:
con.close()
print("Read-only SQLite connection closed.")


Read-only SQLite connection closed.
